# Tuned Lens: Learning Per-Layer Transforms

**Survey Reference:** Section 9.2 - Tuned Lens: Learning Per-Layer Transform

## Overview

The **Tuned Lens** improves on the logit lens by learning an affine transformation for each layer that maps intermediate representations to the final layer's format.

### The Key Insight

Early layers may represent the same information in a different "format". The tuned lens learns:

$$\text{TunedLens}(h_l) = (A_l h_l + b_l) \cdot W_U$$

where $A_l, b_l$ are learned per-layer parameters.

## Learning Objectives

1. Understand why learned transforms help
2. Train tuned lens probes
3. Compare logit lens vs tuned lens
4. Interpret what the learned transforms reveal

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. The Tuned Lens Architecture

In [ ]:
class TunedLens(nn.Module):
    """
    Tuned Lens: Learns per-layer affine transforms.
    
    For each layer, learns A_l and b_l such that:
    A_l @ h_l + b_l ≈ h_L (final hidden state)
    """
    
    def __init__(self, n_layers: int, hidden_size: int):
        super().__init__()
        self.n_layers = n_layers
        self.hidden_size = hidden_size
        
        # Per-layer affine transforms
        # Initialize close to identity
        self.transforms = nn.ModuleList([
            nn.Linear(hidden_size, hidden_size)
            for _ in range(n_layers)
        ])
        
        # Initialize to identity-like
        for layer in self.transforms:
            nn.init.eye_(layer.weight)
            nn.init.zeros_(layer.bias)
    
    def forward(self, hidden_states: list[torch.Tensor], layer_idx: int) -> torch.Tensor:
        """
        Apply tuned lens to a specific layer's hidden state.
        """
        h = hidden_states[layer_idx]
        return self.transforms[layer_idx](h)
    
    def decode_all_layers(self, hidden_states: list[torch.Tensor], 
                          model) -> torch.Tensor:
        """
        Apply tuned lens to all layers and project to vocab.
        
        Returns: [n_layers, seq_len, vocab_size] logits
        """
        ln_f = model.transformer.ln_f
        unembed = model.lm_head.weight
        
        all_logits = []
        for layer_idx in range(len(hidden_states)):
            if layer_idx < self.n_layers:
                h_transformed = self.transforms[layer_idx](hidden_states[layer_idx])
            else:
                h_transformed = hidden_states[layer_idx]
            
            h_normed = ln_f(h_transformed)
            logits = h_normed @ unembed.T
            all_logits.append(logits[0])
        
        return torch.stack(all_logits)

## 2. Training the Tuned Lens

We train each layer's transform to minimize the difference between:
- The transformed intermediate representation
- The final layer's representation

In [ ]:
def train_tuned_lens(
    model,
    tokenizer,
    tuned_lens: TunedLens,
    train_texts: list[str],
    n_epochs: int = 10,
    lr: float = 1e-3,
):
    """
    Train tuned lens on a corpus of text.
    
    Each layer's transform is trained to predict the final hidden state.
    """
    optimizer = torch.optim.Adam(tuned_lens.parameters(), lr=lr)
    history = []
    
    for epoch in range(n_epochs):
        epoch_losses = []
        
        for text in tqdm(train_texts, desc=f"Epoch {epoch+1}"):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs, output_hidden_states=True)
            
            hidden_states = outputs.hidden_states
            target = hidden_states[-1]  # Final layer
            
            # Train each layer's transform
            total_loss = 0
            for layer_idx in range(len(hidden_states) - 1):
                h = hidden_states[layer_idx]
                h_transformed = tuned_lens.transforms[layer_idx](h)
                loss = F.mse_loss(h_transformed, target)
                total_loss = total_loss + loss
            
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            epoch_losses.append(total_loss.item())
        
        avg_loss = np.mean(epoch_losses)
        history.append(avg_loss)
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.6f}")
    
    return history

In [ ]:
# Load model
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

# Create tuned lens
n_layers = model.config.n_layer + 1  # +1 for embedding layer
tuned_lens = TunedLens(n_layers, model.config.n_embd).to(device)

In [ ]:
# Sample training texts (in practice, use more data)
train_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "In a hole in the ground there lived a hobbit.",
    "It was the best of times, it was the worst of times.",
    "The capital of France is Paris, a beautiful city.",
    "Machine learning models can learn complex patterns.",
] * 10  # Repeat for more training data

# Train (in practice, use more epochs and data)
# history = train_tuned_lens(model, tokenizer, tuned_lens, train_texts, n_epochs=5)

## 3. Comparing Logit Lens vs Tuned Lens

In [ ]:
def compare_lenses(model, tokenizer, tuned_lens, text: str):
    """
    Compare logit lens and tuned lens on the same input.
    """
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        
        # Logit lens
        ln_f = model.transformer.ln_f
        unembed = model.lm_head.weight
        
        logit_lens_probs = []
        tuned_lens_probs = []
        
        for layer_idx, h in enumerate(hidden_states):
            # Logit lens
            h_normed = ln_f(h)
            ll_logits = h_normed @ unembed.T
            ll_probs = F.softmax(ll_logits[0, -1], dim=-1)
            logit_lens_probs.append(ll_probs)
            
            # Tuned lens
            if layer_idx < tuned_lens.n_layers:
                h_transformed = tuned_lens.transforms[layer_idx](h)
                h_t_normed = ln_f(h_transformed)
                tl_logits = h_t_normed @ unembed.T
                tl_probs = F.softmax(tl_logits[0, -1], dim=-1)
                tuned_lens_probs.append(tl_probs)
            else:
                tuned_lens_probs.append(ll_probs)
    
    return logit_lens_probs, tuned_lens_probs

# TODO: Compare on test example
# ll_probs, tl_probs = compare_lenses(model, tokenizer, tuned_lens, "The capital of France is")

## 4. Analyzing the Learned Transforms

The learned transforms reveal how representations change across layers.

In [ ]:
def analyze_transforms(tuned_lens: TunedLens):
    """
    Analyze the learned affine transforms.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Measure deviation from identity
    deviations = []
    biases = []
    
    for layer_idx, transform in enumerate(tuned_lens.transforms):
        W = transform.weight.detach().cpu()
        b = transform.bias.detach().cpu()
        
        # Frobenius norm of (W - I)
        identity = torch.eye(W.shape[0])
        deviation = (W - identity).norm().item()
        deviations.append(deviation)
        
        # Bias norm
        biases.append(b.norm().item())
    
    axes[0].plot(deviations, marker='o')
    axes[0].set_xlabel('Layer')
    axes[0].set_ylabel('||W - I||')
    axes[0].set_title('Deviation from Identity')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(biases, marker='o', color='orange')
    axes[1].set_xlabel('Layer')
    axes[1].set_ylabel('||b||')
    axes[1].set_title('Bias Magnitude')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# TODO: Analyze after training
# analyze_transforms(tuned_lens)

## 5. Entropy-Based Analysis

Compare the entropy of predictions across layers for both lenses.

In [ ]:
def compute_entropy(probs: torch.Tensor) -> float:
    """Compute entropy of a probability distribution."""
    log_probs = torch.log(probs + 1e-10)
    return -(probs * log_probs).sum().item()

def entropy_analysis(logit_lens_probs, tuned_lens_probs):
    """
    Compare entropy across layers for both lenses.
    """
    ll_entropies = [compute_entropy(p) for p in logit_lens_probs]
    tl_entropies = [compute_entropy(p) for p in tuned_lens_probs]
    
    plt.figure(figsize=(10, 5))
    plt.plot(ll_entropies, marker='o', label='Logit Lens')
    plt.plot(tl_entropies, marker='s', label='Tuned Lens')
    plt.xlabel('Layer')
    plt.ylabel('Entropy (nats)')
    plt.title('Prediction Entropy: Logit Lens vs Tuned Lens')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# TODO: Run after training
# entropy_analysis(ll_probs, tl_probs)

## Key Takeaways

1. **Tuned lens** learns per-layer affine transforms for better interpretability
2. **Earlier layers** typically need larger transforms (more deviation from identity)
3. **Lower entropy** predictions with tuned lens indicate better calibration
4. **Training is simple:** Just predict final layer from each intermediate layer
5. **Reveals representation drift:** How much the "format" changes across layers

## Next Steps

→ **03_patchscopes.ipynb**: Use the model itself to explain intermediate representations